# Batched generation & padding side

Compare HuggingFace `generate` outputs across three strategies on the
same Qwen2.5-3B-Instruct model:

1. **Case 1** — sequential generation, one prompt at a time
   (reference).
2. **Case 2** — batched generation with **left** padding (correct
   for causal LMs).
3. **Case 3** — batched generation with **right** padding
   (incorrect; for contrast).

Companion notebook
`test_transformers_batched_prompt_scoring_v1.ipynb` covers
log-probability and last-token-embedding extraction under padding.

## Setup

In [1]:
import gc

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Load tokenizer and causal LM onto GPU 0 in eval mode.
# torch_dtype="auto" picks up Qwen2.5's bf16 weights (vs. the fp32 default,
# which would double GPU memory for no accuracy gain).
device = "cuda:0"
tokenizer = AutoTokenizer.from_pretrained(llm_dir)
llm_tf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    torch_dtype="auto",
    device_map=device,
)
llm_tf.eval()

gc.collect()
torch.cuda.empty_cache()
print('#--- memory:', torch.cuda.memory_allocated(0) / (1024**3))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

#--- memory: 5.74800968170166


## Test prompts

In [4]:
# Sample prompts of varying lengths to exercise padding behavior
texts = [
    "Hello, how are you?",
    "What is your name?",
    "Tell me a joke.",
    "Explain quantum computing in simple terms."
]

## Generation: padding strategies

For each strategy below, `generate` is called on the same prompts with the
same seed. Compare the outputs: Case 1 and Case 2 should agree; Case 3
should differ (and look worse) on the shorter prompts that get right-padded.

In [ ]:
# Case 1: Sequential generation (no batching, no padding).
# Reference behavior - each prompt is generated independently.
seed = 100000 + 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

completions_no_padding = []
for text in texts:
    inputs = tokenizer(text, return_tensors="pt").to(llm_tf.device)
    with torch.no_grad():
        output_tokens = llm_tf.generate(
            **inputs,
            max_new_tokens=20,
            pad_token_id=tokenizer.pad_token_id,
        )

    completions_no_padding.append(tokenizer.decode(output_tokens[0]))

for i, (prompt, completion) in enumerate(zip(texts, completions_no_padding)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

Input 1:  Hello, how are you?
Output 1: Hello, how are you? I hope you're doing well. How can I assist you today?
Hello! I'm doing pretty

Input 2:  What is your name?
Output 2: What is your name? My name is Max. What is the weather like today? I'm sorry, as an AI model

Input 3:  Tell me a joke.
Output 3: Tell me a joke. Why was the math book sad? Because it had too many problems.<|endoftext|>

Input 4:  Explain quantum computing in simple terms.
Output 4: Explain quantum computing in simple terms. Quantum computing is a type of computing where information is processed using quantum bits, or qubits, which



In [ ]:
# Case 2: Batched generation with LEFT padding.
# Left padding is the correct choice for causal LM generation: real tokens stay
# right-aligned, so the model conditions on the last real token when emitting
# new tokens. Continuations should match Case 1 (modulo numerical drift).
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Set padding side explicitly - the tokenizer is shared state and any prior
# cell may have left it on the wrong side.
tokenizer.padding_side = 'left'
inputs = tokenizer(texts, return_tensors="pt", padding=True).to(llm_tf.device)

with torch.no_grad():
    output_tokens = llm_tf.generate(
        **inputs,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
    )

completions = tokenizer.batch_decode(output_tokens)
assert len(completions) == len(texts)

for i, (prompt, completion) in enumerate(zip(texts, completions)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

Input 1:  Hello, how are you?
Output 1: <|endoftext|><|endoftext|>Hello, how are you? How can I help you today?

As an AI language model, I don't have feelings, but

Input 2:  What is your name?
Output 2: <|endoftext|><|endoftext|><|endoftext|>What is your name? 
My name is Andrew. 

Nice to meet you! Could you tell me a little bit more

Input 3:  Tell me a joke.
Output 3: <|endoftext|><|endoftext|><|endoftext|>Tell me a joke. Why don't scientists trust atoms? Because they make up everything.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>

Input 4:  Explain quantum computing in simple terms.
Output 4: Explain quantum computing in simple terms. Sure! Imagine you have a special kind of calculator that can do things regular calculators can't.



In [ ]:
# Case 3: Batched generation with RIGHT padding.
# Right padding misaligns shorter prompts during generation - pads sit between
# the prompt and the new tokens, which typically degrades output quality.
# Included for comparison against Case 2.
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Set padding side explicitly - the tokenizer is shared state and any prior
# cell may have left it on the wrong side.
tokenizer.padding_side = 'right'
inputs = tokenizer(texts, return_tensors="pt", padding=True).to(llm_tf.device)

with torch.no_grad():
    output_tokens = llm_tf.generate(
        **inputs,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
    )

completions = tokenizer.batch_decode(output_tokens)
assert len(completions) == len(texts)

for i, (prompt, completion) in enumerate(zip(texts, completions)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Input 1:  Hello, how are you?
Output 1: Hello, how are you?<|endoftext|><|endoftext|> I'm new to python and I need help with a problem I have. I have a list of

Input 2:  What is your name?
Output 2: What is your name?<|endoftext|><|endoftext|><|endoftext|> the most memorable moment in life?
As an artificial intelligence language model, I don't have personal experiences

Input 3:  Tell me a joke.
Output 3: Tell me a joke.<|endoftext|><|endoftext|><|endoftext|> joke about lawyers and cookies.
Why did the cookie go to the doctor?
Because it had a chocolate

Input 4:  Explain quantum computing in simple terms.
Output 4: Explain quantum computing in simple terms. Sure! Imagine you have a special kind of calculator that can do things regular calculators can't.



In [8]:
print(llm_tf.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

